# Defense Evaluation: Watermark, Validation, Proactive, Composite

This notebook evaluates four defense mechanisms against the three attack methodologies:

- **Watermark Defense**: Unigram z-score detection (Zhao et al., ICLR 2024, arXiv:2306.17439)
- **Content Validation Defense**: Pattern-based adversarial content filtering
- **Proactive Defense**: Pre-emptive attack simulation and vulnerability assessment
- **Composite Defense**: Multi-layer combination of all three defenses

**Defense Metric Definitions**:
- **TPR** (True Positive Rate): Fraction of adversarial entries correctly detected
- **FPR** (False Positive Rate): Fraction of benign entries incorrectly flagged
- **F1**: Harmonic mean of precision and recall
- **Utility**: Fraction of benign queries answered correctly despite defense

In [ ]:
import sys
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

from evaluation.benchmarking import BenchmarkRunner, AttackEvaluator, DefenseEvaluator
from evaluation.retrieval_sim import RetrievalSimulator
from attacks.implementations import AttackSuite
from defenses.implementations import create_defense
from data.synthetic_corpus import SyntheticCorpus

matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['font.size'] = 11
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

SEED = 42
CORPUS_SIZE = 200
print('defense evaluation setup complete')

## 1. Generate Poisoned Content Samples

In [ ]:
corpus = SyntheticCorpus(seed=SEED)
benign_entries = corpus.generate_benign_entries(50)
clean_content = [e['content'] for e in benign_entries]

# generate poisoned content via attack suite
attack_suite = AttackSuite()
poisoned_content = []

for entry in benign_entries[:20]:
    results = attack_suite.execute_all(entry['content'])
    for at, result in results['attack_results'].items():
        pc = (
            result.get('poisoned_content')
            or result.get('injected_content')
            or result.get('manipulated_content')
        )
        if pc and isinstance(pc, str):
            poisoned_content.append(pc)

print(f'clean content samples: {len(clean_content)}')
print(f'poisoned content samples: {len(poisoned_content)}')
print(f'\nsample clean: {clean_content[0][:80]}')
print(f'sample poisoned: {poisoned_content[0][:80] if poisoned_content else "n/a"}')

## 2. Evaluate Each Defense

In [ ]:
defense_evaluator = DefenseEvaluator()
defense_types = ['watermark', 'validation', 'proactive', 'composite']

defense_results = {}
for dt in defense_types:
    m = defense_evaluator.evaluate_defense(
        dt, attack_suite, clean_content, poisoned_content
    )
    defense_results[dt] = m
    print(f'\n{dt}:')
    print(f'  TPR={m.tpr:.3f}  FPR={m.fpr:.3f}  Precision={m.precision:.3f}  F1={m.f1_score:.3f}')
    print(f'  TP={m.true_positives}  FP={m.false_positives}  TN={m.true_negatives}  FN={m.false_negatives}')

## 3. Defense Performance Bar Chart (TPR, FPR, F1)

In [ ]:
defense_labels = {
    'watermark': 'Watermark\n(Zhao et al.)',
    'validation': 'Content\nValidation',
    'proactive': 'Proactive\nDefense',
    'composite': 'Composite\n(Multi-layer)',
}

x = np.arange(len(defense_types))
width = 0.25
colors = {'tpr': '#27AE60', 'fpr': '#E74C3C', 'f1_score': '#3498DB'}

fig, ax = plt.subplots(figsize=(12, 6))

for i, (metric, color, label) in enumerate([
    ('tpr', '#27AE60', 'TPR (Detection Rate)'),
    ('fpr', '#E74C3C', 'FPR (False Positive Rate)'),
    ('f1_score', '#3498DB', 'F1 Score'),
]):
    values = [getattr(defense_results[dt], metric) for dt in defense_types]
    bars = ax.bar(x + (i - 1) * width, values, width, color=color, alpha=0.85,
                  edgecolor='black', linewidth=0.5, label=label)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([defense_labels[dt] for dt in defense_types])
ax.set_ylabel('Score')
ax.set_title('Defense Performance Metrics: TPR, FPR, and F1 Score', fontsize=13)
ax.set_ylim(0, 1.15)
ax.legend(loc='upper right')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('../visualization/fig_defense_performance_comparison.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_defense_performance_comparison.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: fig_defense_performance_comparison')

## 4. ROC Curve: Defense Trade-off Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

defense_colors = {
    'watermark': '#8E44AD',
    'validation': '#E67E22',
    'proactive': '#16A085',
    'composite': '#C0392B',
}
defense_markers = {'watermark': 'o', 'validation': 's', 'proactive': '^', 'composite': 'D'}

# random classifier baseline
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, linewidth=1, label='Random classifier')
ax.fill_between([0, 1], [0, 1], [1, 1], alpha=0.05, color='green')
ax.text(0.05, 0.95, 'Ideal region', color='darkgreen', fontsize=9, alpha=0.7)

for dt in defense_types:
    m = defense_results[dt]
    ax.scatter(
        m.fpr, m.tpr,
        c=defense_colors[dt],
        marker=defense_markers[dt],
        s=180,
        zorder=5,
        label=f'{defense_labels[dt].replace(chr(10), " ")} (F1={m.f1_score:.2f})',
        edgecolors='black',
        linewidths=1,
    )

ax.set_xlabel('False Positive Rate (FPR)', fontsize=12)
ax.set_ylabel('True Positive Rate (TPR)', fontsize=12)
ax.set_title('Defense ROC Space: Effectiveness vs False Positive Trade-off', fontsize=12)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.legend(loc='lower right', fontsize=9)
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('../visualization/fig_defense_roc_space.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_defense_roc_space.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: fig_defense_roc_space')

## 5. Attack-Defense Interaction Matrix

In [ ]:
# run each attack then evaluate each defense's effectiveness against those specific attack outputs
attack_types = ['agent_poison', 'minja', 'injecmem']
attack_labels = {'agent_poison': 'AgentPoison', 'minja': 'MINJA', 'injecmem': 'InjecMEM'}

interaction_matrix = np.zeros((len(defense_types), len(attack_types)))

for j, at in enumerate(attack_types):
    # generate poisoned content specific to this attack
    atk = attack_suite.get_attack(at)
    attack_poison = []
    for entry in benign_entries[:15]:
        result = atk.execute(entry['content'])
        pc = (
            result.get('poisoned_content')
            or result.get('injected_content')
            or result.get('manipulated_content')
        )
        if pc and isinstance(pc, str):
            attack_poison.append(pc)
    
    for i, dt in enumerate(defense_types):
        m = defense_evaluator.evaluate_defense(dt, attack_suite, clean_content[:15], attack_poison)
        # effectiveness = TPR - FPR (youden index)
        interaction_matrix[i, j] = m.tpr - m.fpr

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(interaction_matrix, cmap='RdYlGn', vmin=-0.5, vmax=1.0, aspect='auto')
ax.set_xticks(range(len(attack_types)))
ax.set_xticklabels([attack_labels[at] for at in attack_types])
ax.set_yticks(range(len(defense_types)))
ax.set_yticklabels([defense_labels[dt].replace('\n', ' ') for dt in defense_types])
ax.set_title('Attack-Defense Interaction Matrix\n(Youden Index: TPR - FPR; Higher is Better Defense)', fontsize=11)
plt.colorbar(im, ax=ax, label='TPR - FPR (Youden Index)')

for i in range(len(defense_types)):
    for j in range(len(attack_types)):
        ax.text(j, i, f'{interaction_matrix[i, j]:.2f}', ha='center', va='center',
                fontsize=11, fontweight='bold',
                color='white' if abs(interaction_matrix[i, j]) > 0.5 else 'black')

plt.tight_layout()
plt.savefig('../visualization/fig_attack_defense_interaction_matrix.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_attack_defense_interaction_matrix.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: fig_attack_defense_interaction_matrix')

## 6. Defense Impact on ASR-R (Retrieval Reduction)

In [ ]:
# compute how much each defense reduces ASR-R
# modelled as: post-defense ASR-R = pre-defense ASR-R * (1 - detection_accuracy)
# where detection_accuracy = TPR of the defense against that attack's poison

sim = RetrievalSimulator(corpus_size=CORPUS_SIZE, top_k=5, n_poison_per_attack=5, seed=SEED)

pre_asr_r = {}
for at in attack_types:
    m = sim.evaluate_attack(at)
    pre_asr_r[at] = m.asr_r

# for each defense, compute approximate post-defense ASR-R
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(attack_types))
total_width = 0.7
bar_width = total_width / (len(defense_types) + 1)

# plot pre-defense (baseline)
bars = ax.bar(x - total_width/2, [pre_asr_r[at] for at in attack_types],
              bar_width, color='#7F8C8D', alpha=0.9, edgecolor='black',
              linewidth=0.5, label='No Defense (Baseline)')

defense_bar_colors = {'watermark': '#8E44AD', 'validation': '#E67E22', 'proactive': '#16A085', 'composite': '#C0392B'}

for di, dt in enumerate(defense_types):
    dm = defense_results[dt]
    post_asr_values = []
    for at in attack_types:
        # reduced asr_r = pre * (1 - tpr) (poison that evades detection)
        post = pre_asr_r[at] * (1 - dm.tpr)
        post_asr_values.append(post)
    
    bars = ax.bar(x - total_width/2 + (di+1)*bar_width, post_asr_values,
                  bar_width, color=defense_bar_colors[dt], alpha=0.85,
                  edgecolor='black', linewidth=0.5,
                  label=defense_labels[dt].replace('\n', ' '))

ax.set_xticks(x)
ax.set_xticklabels([attack_labels[at] for at in attack_types])
ax.set_ylabel('ASR-R (Estimated Post-Defense)')
ax.set_title('Defense Impact on Retrieval Attack Success Rate (ASR-R)', fontsize=12)
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig('../visualization/fig_defense_asr_reduction.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_defense_asr_reduction.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: fig_defense_asr_reduction')

## 7. Save Defense Evaluation Results

In [ ]:
# tpr degradation curves: all three attacks on one plot
evasion_colors = {'paraphrase': '#E74C3C', 'copy_paste': '#F39C12', 'adaptive_substitution': '#8E44AD'}
evasion_labels = {
    'paraphrase': 'Paraphrasing (Synonym Substitution)',
    'copy_paste': 'Copy-Paste Dilution',
    'adaptive_substitution': 'Adaptive Substitution (White-Box)',
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Watermark Evasion: TPR Degradation and Z-Score Distribution Shift\n'
             '(Unigram Watermark, Zhao et al. ICLR 2024)', fontsize=13)

# left: tpr vs intensity
ax = axes[0]
for attack_type, result in evasion_results.items():
    intensities = [r['intensity'] for r in result.intensity_results]
    tprs = [r['tpr'] for r in result.intensity_results]
    ax.plot(intensities, tprs, marker='o', color=evasion_colors[attack_type],
            linewidth=2, markersize=7, label=evasion_labels[attack_type])

ax.axhline(y=evasion_results['paraphrase'].tpr_before, color='black', linestyle='--',
           linewidth=1.5, alpha=0.6, label=f'Baseline TPR = {evasion_results["paraphrase"].tpr_before:.3f}')
ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5, linewidth=1)
ax.set_xlabel('Attack Intensity')
ax.set_ylabel('True Positive Rate (TPR)')
ax.set_title('TPR Degradation vs. Evasion Intensity')
ax.legend(fontsize=9, loc='lower left')
ax.set_ylim(-0.05, 1.1)
ax.grid(True, alpha=0.2)
ax.text(0.02, 0.52, 'Detection threshold (TPR=0.5)', fontsize=8, color='gray')

# right: z-score distributions before and after evasion
ax = axes[1]
import statistics as stats_mod
for attack_type, result in evasion_results.items():
    if result.z_scores_before:
        ax.scatter(
            [evasion_labels[attack_type]] * len(result.z_scores_before),
            result.z_scores_before,
            alpha=0.3, color=evasion_colors[attack_type], marker='o', s=20
        )
    if result.z_scores_after:
        ax.scatter(
            [evasion_labels[attack_type]] * len(result.z_scores_after),
            result.z_scores_after,
            alpha=0.7, color=evasion_colors[attack_type], marker='x', s=50
        )

# reference line: detection threshold
z_threshold = getattr(encoder, 'z_threshold', 4.0)
ax.axhline(y=z_threshold, color='red', linestyle='--', linewidth=1.5,
           alpha=0.8, label=f'z-threshold = {z_threshold:.1f}')
ax.set_ylabel('Z-Score')
ax.set_title('Z-Score Before (o) and After (x) Evasion\nat Maximum Intensity')
ax.tick_params(axis='x', rotation=25)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.savefig('../visualization/fig_watermark_evasion_tpr_degradation.png', dpi=300, bbox_inches='tight')
plt.savefig('../visualization/fig_watermark_evasion_tpr_degradation.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: fig_watermark_evasion_tpr_degradation')

In [ ]:
# run all three evasion evaluations
ev = WatermarkEvasionEvaluator(encoder, n_samples=20, seed=SEED)
evasion_results = ev.evaluate_all(wm_samples, clean_samples)
evasion_report = ev.generate_evasion_report(evasion_results)

print('=== evasion evaluation summary ===')
for attack_type, result in evasion_results.items():
    s = result.summary()
    print(f'\n{attack_type}:')
    print(f'  tpr: {s["tpr_before"]:.3f} → {s["tpr_after"]:.3f}  (delta={s["tpr_delta"]:+.3f})')
    print(f'  evasion success rate: {s["evasion_success_rate"]:.3f}')
    print(f'  mean z-score: {s["mean_z_before"]:.2f} → {s["mean_z_after"]:.2f}')
    if s["mean_semantic_similarity"] > 0:
        print(f'  semantic preservation: {s["mean_semantic_similarity"]:.3f}')

In [ ]:
from watermark.watermarking import UnigramWatermarkEncoder
from evaluation.evasion_eval import WatermarkEvasionEvaluator

encoder = UnigramWatermarkEncoder()

# build watermarked and clean sample sets
wm_content_base = (
    "the memory agent system stores and retrieves contextual information "
    "across multiple interaction sessions. the scheduler confirms task "
    "completion, meeting attendance, and deadline tracking. the system "
    "verifies authentication and validates authorization for all access "
    "requests. confirmed schedule entries require approved protocol steps "
    "for execution and verified credential submission from authorized users. "
    "the configuration settings are stored and retrieved according to the "
    "established preference record and memory entry history database."
)
clean_content_base = [e['content'] for e in corpus.generate_benign_entries(20)]

# embed watermark into each sample (with unique watermark id per entry)
wm_samples = [
    encoder.embed(wm_content_base + f" session {i}.", f"evasion_wm_{i}")
    for i in range(20)
]
clean_samples = clean_content_base[:20]

print(f'watermarked samples: {len(wm_samples)}')
print(f'clean samples: {len(clean_samples)}')

# baseline detection stats
z_scores_wm = [encoder.get_detection_stats(t)["z_score"] for t in wm_samples]
z_scores_cl = [encoder.get_detection_stats(t)["z_score"] for t in clean_samples]
tpr_base = sum(encoder.get_detection_stats(t)["detected"] for t in wm_samples) / len(wm_samples)
fpr_base = sum(encoder.get_detection_stats(t)["detected"] for t in clean_samples) / len(clean_samples)

print(f'\nbaseline detection:')
print(f'  mean z-score (watermarked): {sum(z_scores_wm)/len(z_scores_wm):.2f}')
print(f'  mean z-score (clean):       {sum(z_scores_cl)/len(z_scores_cl):.2f}')
print(f'  tpr (baseline): {tpr_base:.3f}')
print(f'  fpr (baseline): {fpr_base:.3f}')

## 7. Phase 10 Upgrade: Watermark Evasion Evaluation

Three evasion attack classes against the Unigram watermark defense (Zhao et al., ICLR 2024):

1. **Paraphrasing**: Synonym substitution at varying intensity rates — simulates a paraphrase LM
2. **Copy-Paste Dilution**: Mixes watermarked text with non-watermarked filler; theoretical z-score reduction: $z_{diluted} = z_{wm} / \sqrt{1 + r}$
3. **Adaptive Substitution**: White-box attack — greedily replaces green-list tokens with red-list synonyms to drive $z$ below the detection threshold

In [ ]:
from dataclasses import asdict

results_to_save = {
    'experiment': 'defense_evaluation',
    'corpus_size': CORPUS_SIZE,
    'n_clean': len(clean_content),
    'n_poisoned': len(poisoned_content),
    'defense_metrics': {dt: asdict(defense_results[dt]) for dt in defense_types},
    'pre_defense_asr_r': pre_asr_r,
}

output_path = '../visualization/defense_evaluation_results.json'
with open(output_path, 'w') as f:
    json.dump(results_to_save, f, indent=2)

print(f'results saved to {output_path}')
print('\n=== Defense Summary ===')
for dt in defense_types:
    m = defense_results[dt]
    print(f'{dt:15s}: TPR={m.tpr:.3f}  FPR={m.fpr:.3f}  F1={m.f1_score:.3f}')